In [69]:
import pandas as pd             
import mysql.connector         
from getpass import getpass    

In [70]:
# dtype=str en nregistro y cn: son codigos identificadores, no son números con los que operar.
df = pd.read_csv(
    "../data/HU5_medicamentos_diabetes_clusters.csv",
    encoding="utf-8-sig",
    dtype={"nregistro": str, "cn": str}
)

In [71]:
print(df.shape)
df.head()

(827, 30)


,nregistro,nombre,pactivos,labtitular,labcomercializador,cn,dosis,forma_farmaceutica_simplificada,estado_aut,comercializado,...,estado,precio_pvp_iva,tratamiento_larga_duracion,tiene_revision,tiene_foto_materiales,dosis_por_ml,dosis_numero_2,dosis_numero,cluster,perfil
0,70452,ACARBOSA TECNIGEN 100 mg COMPRIMIDOS,ACARBOSA,Tecnimede España Industria Farmaceutica S.A.,Tecnimede España Industria Farmaceutica S.A.,662260,100 mg,COMPRIMIDO,2009-02-04 23:00:00,1,...,ALTA,15.30,1,0,1,0,0.0,100.0,3,Baratos de larga duración
1,70451,ACARBOSA TECNIGEN 50 mg COMPRIMIDOS,ACARBOSA,Tecnimede España Industria Farmaceutica S.A.,Tecnimede España Industria Farmaceutica S.A.,662258,50 mg,COMPRIMIDO,2009-02-04 23:00:00,1,...,ALTA,7.65,1,0,1,0,0.0,50.0,3,Baratos de larga duración
2,2230011,ACTRAPID INNOLET 100 UI/ML SOLUCION INYECTABLE...,INSULINA SOLUBLE NEUTRA,Novo Nordisk A/S,Novo Nordisk Pharma S.A.,775536,3.5 mg,INYECTABLE,2000-11-05 23:00:00,0,...,BAJA POR NO COMERCIALIZAR,35.89,1,0,1,1,0.0,3.5,5,Inyectables no genéricos
3,89509,BIGETRA 150 MG CAPSULAS DURAS,DABIGATRAN ETEXILATO MESILATO,Adamed Laboratorios S.L.U.,Adamed Laboratorios S.L.U.,764780,150 mg,CAPSULA,2024-04-22 09:44:08,0,...,BAJA GENERAL,45.08,0,1,1,0,0.0,150.0,4,Anticoagulantes con ficha compleja
4,111696003,BYDUREON 2 MG POLVO Y DISOLVENTE PARA SUSPENSI...,EXENATIDA,Astrazeneca Ab,Astrazeneca Farmaceutica Spain S.A.,703897,2 mg,INYECTABLE,2014-10-19 22:00:00,0,...,BAJA GENERAL,93.34,0,1,1,0,0.0,2.0,5,Inyectables no genéricos


In [72]:
# Datos de conexión
config = {
    "host": "localhost",    
    "port": 3306,            
    "user": "root",
    "database": "muiaap_2026",
    # para que se guarden bien las ñ y las tildes
    "charset": "utf8mb4",
}

# Abrimos la conexión pidiendo la contraseña (así no queda escrita en el cuaderno)
conn = mysql.connector.connect(password=getpass("Contraseña: "), **config)

# buffered=True: el cursor descarga y guarda todos los resultados en cuanto ejecuta la consulta.
cursor = conn.cursor(buffered=True)


print("Conectado:", conn.is_connected())

Conectado: True


## Creacion de tabla

In [73]:
# Primero hacemos un drop table por si la tabla ya existe
cursor.execute("DROP TABLE IF EXISTS medicamentos_diabetes")

# Ponemos una columna SQL por cada columna del CSV, en el mismo orden, y su respectivo tipo de dat.
create_sql = """
CREATE TABLE medicamentos_diabetes (
    nregistro                       VARCHAR(20) PRIMARY KEY,
    nombre                          VARCHAR(500),
    pactivos                        VARCHAR(500),
    labtitular                      VARCHAR(255),
    labcomercializador              VARCHAR(255),
    cn                              VARCHAR(10),
    dosis                           VARCHAR(100),
    forma_farmaceutica_simplificada VARCHAR(50),
    estado_aut                      DATETIME,
    comercializado                  INT,
    generico                        INT,
    afecta_conduccion               INT,
    biosimilar                      INT,
    num_registros_atc               INT,
    num_principios_activos          INT,
    num_excipientes                 INT,
    vias_administracion             VARCHAR(50),
    palabras_seccion_4_4            INT,
    numero_tablas                   INT,
    indicador_riesgo_severo         INT,
    estado                          VARCHAR(100),
    precio_pvp_iva                  DOUBLE,
    tratamiento_larga_duracion      INT,
    tiene_revision                  INT,
    tiene_foto_materiales           INT,
    dosis_por_ml                    INT,
    dosis_numero_2                  DOUBLE,
    dosis_numero                    DOUBLE,
    cluster                         INT,
    perfil                          VARCHAR(100)
)
"""
cursor.execute(create_sql)
print("Tabla creada")

Tabla creada


In [74]:
# Preparamos la sentencia INSERT con tantos %s como columnas (30).
# Los %s son "huecos" que el conector rellena con cada valor de forma segura.
huecos = ", ".join(["%s"] * len(df.columns))
insert_sql = f"INSERT INTO medicamentos_diabetes VALUES ({huecos})"

# Convertimos el DataFrame en una lista de filas.
# astype(object) + tolist() convierte los números de numpy en números normales de Python,
# que es lo que entiende mysql.connector.
filas = df.astype(object).values.tolist()

# Executemany inserta todas las filas de golpe.
cursor.executemany(insert_sql, filas)

# Hacemos commit para que se queden los datos en la base de datos
conn.commit()
print("Filas insertadas:", cursor.rowcount)

Filas insertadas: 827


In [75]:
# Una VISTA es una "consulta guardada": no copia los datos, los calcula al consultarla.
# Ventaja: si cambian los datos de la tabla, el KPI se actualiza solo.
# CREATE OR REPLACE: si la vista ya existe, la sustituye.
vista_sql = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos3 AS
SELECT
    nregistro,
    nombre,
    num_principios_activos,
    num_excipientes,
    num_principios_activos + num_excipientes AS total_componentes   -- KPI
FROM medicamentos_diabetes
"""
cursor.execute(vista_sql)
print("Vista creada")

Vista creada


In [76]:
vista_sql = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos5 AS
SELECT
    nregistro,
    nombre,
    forma_farmaceutica_simplificada,
    precio_pvp_iva,
    num_principios_activos,
    num_excipientes,
    num_principios_activos + num_excipientes AS total_componentes,
    ROW_NUMBER() OVER (
        PARTITION BY forma_farmaceutica_simplificada
        ORDER BY precio_pvp_iva DESC, nregistro
    ) AS ranking_precio_grupo
FROM medicamentos_diabetes
"""
cursor.execute(vista_sql)

In [77]:
vista_sql = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos2 AS
SELECT
    nregistro,
    nombre,
    num_principios_activos,
    num_excipientes,
    num_principios_activos + num_excipientes AS total_componentes,
    palabras_seccion_4_4,
    numero_tablas,
    ROUND(
        palabras_seccion_4_4 / NULLIF(numero_tablas, 0),
        2
    ) AS densidad_ficha_tecnica
FROM medicamentos_diabetes
"""

cursor.execute(vista_sql)
print("Vista actualizada")

Vista actualizada


In [78]:
   #  KPI 1
#1. Desviación de precio respecto a la media del grupo (%): Calcular el porcentaje de diferencia entre el PVP-IVA de cada medicamento
#y el PVP-IVA medio de los medicamentos con la misma forma_farmaceutica_simplificada, mediante la fórmula:
vista_sql = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos1 AS
SELECT
    nregistro,
    nombre,
    forma_farmaceutica_simplificada,
    precio_pvp_iva,
    ROUND(
        100 * (
            precio_pvp_iva
            - AVG(precio_pvp_iva) OVER (
                PARTITION BY forma_farmaceutica_simplificada
            )
        )
        / NULLIF(
            AVG(precio_pvp_iva) OVER (
                PARTITION BY forma_farmaceutica_simplificada
            ),
            0
        ),
        2
    ) AS desviacion_precio_grupo_pct
FROM medicamentos_diabetes;
"""

cursor.execute(vista_sql)
print("Vista actualizada")

Vista actualizada


In [79]:
      ##KPI4

##Diferencia de precio absoluta respecto al mínimo del grupo: Calcular cuánto supera el PVP-IVA de cada medicamento al precio más 
##bajo de los medicamentos con la misma forma_farmaceutica_simplificada.
vista_sql = """
CREATE OR REPLACE VIEW vista_kpi_medicamentos4 AS
SELECT
    nregistro,
    nombre,
    forma_farmaceutica_simplificada,
    precio_pvp_iva,
    ROUND(
        precio_pvp_iva
        - MIN(precio_pvp_iva) OVER (
            PARTITION BY forma_farmaceutica_simplificada
        ),
        2
    ) AS diferencia_precio_minimo_grupo
FROM medicamentos_diabetes;
"""

cursor.execute(vista_sql)
print("Vista actualizada")

Vista actualizada
